In [1]:
import os
os.environ["HF_TOKEN"] = "hf_..."

https://huggingface.co/google/functiongemma-270m-it

In [2]:
from transformers import AutoProcessor, AutoModelForCausalLM

processor = AutoProcessor.from_pretrained("google/functiongemma-270m-it", device_map="auto")
model = AutoModelForCausalLM.from_pretrained("google/functiongemma-270m-it", dtype="auto", device_map="auto")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.32k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.69M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/63.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/706 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/13.8k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/536M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/176 [00:00<?, ?B/s]

In [3]:
import json
import random

def write_txt_file(file_path: str, content: str):
    """
    Write a string into a .txt file (overwrites if exists).
    Args:
        file_path (str): Destination path.
        content (str): Text to write.
    Returns:
        str: Path to the written file.
    """
    with open(file_path, "w", encoding="utf-8") as f:
        f.write(content)
    return file_path


def get_current_temperature(location):
    """
    Mock function to get the current temperature for a city

    Args:
        location (str): City name

    Returns:
        dict: Temperature information for the city
    """
    # Simple database of mock temperatures for several cities
    mock_temperatures = {
        "london": {"temp": 15, "unit": "celsius", "condition": "cloudy"},
        "san francisco": {"temp": 18, "unit": "celsius", "condition": "sunny"},
        "tehran": {"temp": 25, "unit": "celsius", "condition": "clear"},
        "new york": {"temp": 12, "unit": "celsius", "condition": "rainy"},
        "tokyo": {"temp": 20, "unit": "celsius", "condition": "partly cloudy"},
    }

    # Normalize city name
    location_normalized = location.lower().strip()

    # If city is in database
    if location_normalized in mock_temperatures:
        temp_data = mock_temperatures[location_normalized]
        return {
            "location": location,
            "temperature": temp_data["temp"],
            "unit": temp_data["unit"],
            "condition": temp_data["condition"],
            "status": "success"
        }
    else:
        # For other cities, generate random temperature
        return {
            "location": location,
            "temperature": random.randint(10, 30),
            "unit": "celsius",
            "condition": "unknown",
            "status": "success"
        }


get_current_temperature("Tehran")


{'location': 'Tehran',
 'temperature': 25,
 'unit': 'celsius',
 'condition': 'clear',
 'status': 'success'}

In [4]:
weather_function_schema = {
    "type": "function",
    "function": {
        "name": "get_current_temperature",
        "description": "Gets the current temperature for a given location.",
        "parameters": {
            "type": "object",
            "properties": {
                "location": {
                    "type": "string",
                    "description": "The city name, e.g. San Francisco",
                },
            },
            "required": ["location"],
        },
    }
}

write_file_function_schema = {
    "type": "function",
    "function": {
        "name": "write_txt_file",
        "description": "Write a string into a .txt file (overwrites if exists).",
        "parameters": {
            "type": "object",
            "properties": {
                "file_path": {
                    "type": "string",
                    "description": "Destination path for the .txt file, e.g. 'output/report.txt'",
                },
                "content": {
                    "type": "string",
                    "description": "Text content to write into the file",
                },
            },
            "required": ["file_path", "content"],
        },
    }
}


In [5]:
message = [
    # ESSENTIAL SYSTEM PROMPT:
    # This line activates the model's function calling logic.
    {
        "role": "developer",
        "content": "You are a model that can do function calling with the following functions"
    },
    {
        "role": "user",
        "content": "What's the temperature in Tehran?"
    }
]

inputs = processor.apply_chat_template(message, tools=[weather_function_schema, write_file_function_schema], add_generation_prompt=True, return_dict=True, return_tensors="pt")

out = model.generate(**inputs.to(model.device), pad_token_id=processor.eos_token_id, max_new_tokens=128)
output = processor.decode(out[0][len(inputs["input_ids"][0]):], skip_special_tokens=True)

print(output)


<start_function_call>call:get_current_temperature{location:<escape>Tehran<escape>}<end_function_call>


In [6]:
# Parse the function call from model output
import re

def parse_function_call(output_text):
    """Extract function name and arguments from model output"""
    # Pattern for FunctionGemma output: <start_function_call>call:function_name{arg:value}<end_function_call>
    pattern = r'<start_function_call>call:(\w+)\{(.+?)\}<end_function_call>'
    match = re.search(pattern, output_text)

    if match:
        function_name = match.group(1)
        args_str = match.group(2)

        # Parse arguments (format: key:<escape>value<escape>)
        args = {}
        arg_pattern = r'(\w+):<escape>(.+?)<escape>'
        for arg_match in re.finditer(arg_pattern, args_str):
            args[arg_match.group(1)] = arg_match.group(2)

        return function_name, args
    return None, None

# Parse the output
function_name, arguments = parse_function_call(output)

print(f"Function to call: {function_name}")
print(f"Arguments: {arguments}")

# Execute the function if parsed successfully
if function_name and arguments:
    if function_name == "get_current_temperature":
        result = get_current_temperature(**arguments)
        print(f"\nFunction result:")
        print(json.dumps(result, indent=2))
    else:
        print(f"Unknown function: {function_name}")
else:
    print("Could not parse function call from output")


Function to call: get_current_temperature
Arguments: {'location': 'Tehran'}

Function result:
{
  "location": "Tehran",
  "temperature": 25,
  "unit": "celsius",
  "condition": "clear",
  "status": "success"
}


In [9]:

message = [
    # ESSENTIAL SYSTEM PROMPT:
    # This line activates the model's function calling logic.
    {
        "role": "developer",
        "content": "You are a model that can do function calling with the following functions"
    },
    {
        "role": "user",
        "content": "Write 'salam' in test.txt"
    }
]

inputs = processor.apply_chat_template(message, tools=[weather_function_schema, write_file_function_schema], add_generation_prompt=True, return_dict=True, return_tensors="pt")

out = model.generate(**inputs.to(model.device), pad_token_id=processor.eos_token_id, max_new_tokens=128)
output = processor.decode(out[0][len(inputs["input_ids"][0]):], skip_special_tokens=True)

print(output)

<start_function_call>call:write_txt_file{content:<escape>salam<escape>,file_path:<escape>test.txt<escape>}<end_function_call>


In [13]:
function_name, arguments = parse_function_call(output)

FUNCTION_REGISTRY = {
    "get_current_temperature": get_current_temperature,
    "write_txt_file": write_txt_file,
}

if not function_name or not arguments:
    print("Could not parse function call from output")
else:
    function = FUNCTION_REGISTRY.get(function_name)

    if not function:
        print(f"Unknown function: {function_name}")
    else:
        result = function(**arguments)
        print("\nFunction result:")
        print(json.dumps(result, indent=2))



Function result:
"test.txt"
